In [3]:
from wazeasy import utils, reports
import dask.dataframe as dd
import pandas as pd
import numpy as np
import yaml
import h3
import geopandas as gpd
from shapely import wkt

import folium
import altair as alt
import attaviz
attaviz.enable()


In [4]:
# %pip install -e ../../wazeasy
# %pip install folium mapclassify matplotlib

# Accidentes
En esta sección del estudio, utilizamos los datos de Alertas de Waze. Estos datos son reportes que generann lo usuarios al utilizar la aplicación. Las alertas pueden ser de diferentes tipos, entre ellas: accidentes, embotellamientos, peligros debido al clima, construcción, etc. 

En nuestro caso, estamos interesados en estudiar los patrones de accidentes, por lo que descartamos el resto de las alertas.

La muestra que estamos utilizando tiene la misma extensión espacial y temporal que la [muestra de embotellamientos](muestra-embotellamientos). 

De los datos, podemos concluir lo siguiente:
- La mayoría de los accidentes ocurren en el área central de Montevideo. 
- La distribución horaria de accidentes para días hábiles es bimodal, con un pico a las 8 AM y otro pico a las 5 PM. 
- La distribución horaria de accidentes durante los fines de semana 

In [5]:
with open("../docs/config_montevideo.yaml", "r") as f:
    config = yaml.safe_load(f)

timezone = config["Montevideo"]["timezone"]["timezone_name"]
s3_path = config["Montevideo"]["s3_path_alerts"]
aws_profile = config["Montevideo"]["aws_profile"]
geographies = config["geographies"]

print("Timezone:", timezone)
print("S3 path:", s3_path)

Timezone: America/Montevideo
S3 path: s3://wbg-waze/bq/UY/784montevideo/alerts/


In [6]:
ddf = dd.read_parquet(s3_path + '*parquet', storage_options={'profile': 'ddp_dec'}, engine='pyarrow')
utils.handle_time(ddf, timezone)
acc = ddf[ddf['type']=='ACCIDENT']
acc = acc.persist()

In [8]:
def assign_h3_alerts(ddf, level=7):
    coords = (
        ddf['geoWKT']
        .str.replace(r'(?i)point\s*\(|\)', '', regex=True)
        .str.strip()
        .str.split(' ', n=1, expand=True)   # Dask needs n= when expand=True
            )
    ddf['lon'] = dd.to_numeric(coords[0], errors='coerce')
    ddf['lat'] = dd.to_numeric(coords[1], errors='coerce')
    ddf["hex_id_{}".format(level)] = ddf.apply(lambda row: h3.latlng_to_cell(row["lat"], row["lon"], level), 
                          axis=1,meta=("hex_id", "string[pyarrow]"))

In [9]:
assign_h3_alerts(acc, level=7)

In [10]:
acc_hex = acc.groupby('hex_id_7').size().compute()

In [11]:
h3_7 = gpd.read_file(geographies['hex_level_7']['path']).set_index('hex_id')

In [12]:
h3_7['Accidentes'] = acc_hex

In [12]:
h3_7[h3_7['Accidentes']>10].explore(column = 'Accidentes', scheme = 'naturalbreaks', cmap = 'Spectral_r')

## Accidentes por Hora

In [ ]:
accidents_by_hour = acc.groupby('hour').size().reset_index().compute()
accidents_by_hour.rename(columns = {0:'Number of Accidents'}, inplace = True)

In [14]:
chart = (
    alt.Chart(accidents_by_hour)
    .mark_bar()
    .encode(
        x=alt.X("hour:O", title="hour", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("Number of Accidents:Q", title="Accidentes"),
        tooltip=["hour", "Number of Accidents"]
    )
    .properties(
        title="Distribución horaria de accidentes (Sep-21 to Ago-26)",
        width=600,
        height=300
    )
    .configure_view(strokeWidth=0)   # quita el borde del gráfico (similar a sns.despine)
    .configure_axis(domain=False)    # quita las líneas de los ejes x e y
)

chart


alt.Chart(...)

In [15]:
dates_week = utils.define_dates_of_interest(acc, None, None, dow = [0, 1, 2, 3, 4])
dates_weekend = utils.define_dates_of_interest(acc, None, None, dow = [5,6])

In [16]:
acc_week = acc[acc['date'].isin(dates_week)]
acc_week_by_hour = acc_week.groupby('hour').size().reset_index().compute()
acc_week_by_hour.rename(columns = {0:'Number of Accidents'}, inplace = True)

chart = (
    alt.Chart(acc_week_by_hour)
    .mark_bar()
    .encode(
        x=alt.X("hour:O", title="hour", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("Number of Accidents:Q", title="Accidentes"),
        tooltip=["hour", "Number of Accidents"]
    )
    .properties(
        title="Distribución horaria de accidentes (Sep-21 to Ago-26) | Días Hábiles",
        width=600,
        height=300
    )
    .configure_view(strokeWidth=0)   # quita el borde del gráfico (similar a sns.despine)
    .configure_axis(domain=False)    # quita las líneas de los ejes x e y
)

chart

alt.Chart(...)

In [17]:
acc_weekend = acc[acc['date'].isin(dates_weekend)]
acc_weekend_by_hour = acc_weekend.groupby('hour').size().reset_index().compute()
acc_weekend_by_hour.rename(columns = {0:'Number of Accidents'}, inplace = True)

chart = (
    alt.Chart(acc_weekend_by_hour)
    .mark_bar()
    .encode(
        x=alt.X("hour:O", title="hour", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("Number of Accidents:Q", title="Accidentes"),
        tooltip=["hour", "Number of Accidents"]
    )
    .properties(
        title="Distribución horaria de accidentes (Sep-21 to Ago-26) | Días Hábiles",
        width=600,
        height=300
    )
    .configure_view(strokeWidth=0)   # quita el borde del gráfico (similar a sns.despine)
    .configure_axis(domain=False)    # quita las líneas de los ejes x e y
)

chart

alt.Chart(...)